# 04d — Bogotá CO network series

Per user request. CO is a primary combustion/vehicle-emissions tracer (like NO), and a
classic O3 precursor via the HO2/peroxy-radical cycle (CO + OH -> CO2 + HO2, which then
converts NO to NO2 without consuming O3, a net O3-production pathway distinct from
titration). No definitional-subset relationship with either PM2.5 or O3, so no causal
exclusion needed -- treated the same way NO2/NO were.

Source: `~/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/CO.csv` (same raw
hourly per-station file family as NO.csv/O3.csv, 2013-2022, 20 Bogotá stations). Same
QC convention as notebook 04b (NO): physical range check, Chauvenet's criterion on
hourly values per station, daily mean requiring >=18/24 valid hours, >=70% station-year
completeness, network mean of qualifying stations.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.qc import chauvenet_outlier_mask, completeness_by_station_year, select_stations_years

RAW_PATH = Path("/path/to/home/Documents/Postdoc_MSCA-Bridge/USA_Ozone/DATA_ORIGINAL/CO.csv")
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

raw = pd.read_csv(RAW_PATH)
raw = raw.rename(columns={raw.columns[0]: "datetime"})
raw["datetime"] = pd.to_datetime(raw["datetime"], format="%d/%m/%Y %H:%M", errors="coerce")
raw = raw.dropna(subset=["datetime"])
print(raw.shape, raw["datetime"].min(), raw["datetime"].max())
station_cols = [c for c in raw.columns if c != "datetime"]
print(f"{len(station_cols)} stations:", station_cols)


(87648, 21) 2013-01-01 00:00:00 2022-12-31 23:00:00
20 stations: ['BOLIVIA', 'BOSA', 'CARVAJAL', 'CENTRO DE ALTO RENDIMIENTO', 'CIUDAD BOLIVAR', 'FONTIBON', 'GUAYMARAL', 'JAZMIN', 'KENNEDY', 'LAS FERIAS', 'MIN AMBIENTE', 'PUENTE ARANDA', 'SAN CRISTOBAL', 'SUBA', 'TUNAL', 'USAQUEN', 'USME', 'COLINA ', 'MOVIL FONTIBON', 'MOVIL 7MA']


## Melt to long format, physical range check, Chauvenet outlier flagging

In [2]:
long = raw.melt(id_vars="datetime", value_vars=station_cols, var_name="station", value_name="co")
long["co"] = pd.to_numeric(long["co"], errors="coerce")
print("raw non-null hourly obs:", long["co"].notna().sum())

CO_RANGE = (0.0, 50.0)  # generous -- catches sensor faults, not genuine pollution episodes
bad = (long["co"] < CO_RANGE[0]) | (long["co"] > CO_RANGE[1])
print(f"[range check] co: {int(bad.sum())} values outside {CO_RANGE} -> NaN")
long.loc[bad, "co"] = np.nan

mask = long.groupby("station")["co"].transform(lambda s: chauvenet_outlier_mask(s))
print(f"[Chauvenet] co: flagged {int(mask.sum())} hourly outliers")
long.loc[mask, "co"] = np.nan
print("non-null hourly obs after QC:", long["co"].notna().sum())


raw non-null hourly obs: 702954
[range check] co: 2 values outside (0.0, 50.0) -> NaN
[Chauvenet] co: flagged 2844 hourly outliers
non-null hourly obs after QC: 700108


## Hourly -> daily (per station, requires >=18/24 non-null hours), then completeness filtering

In [3]:
long["date"] = long["datetime"].dt.floor("D")
daily_station = (
    long.groupby(["station", "date"])["co"]
    .agg(co_mean="mean", n_hours="count")
    .reset_index()
)
MIN_HOURS = 18
daily_station.loc[daily_station["n_hours"] < MIN_HOURS, "co_mean"] = np.nan
daily_station = daily_station.rename(columns={"co_mean": "co"})[["station", "date", "co"]]
print(daily_station.shape, "station-days with a valid daily mean:", daily_station["co"].notna().sum())

completeness = completeness_by_station_year(daily_station, value_col="co")
MIN_COVERAGE = 0.70
good_station_years = select_stations_years(completeness, threshold=MIN_COVERAGE)
print(f"{len(good_station_years)} / {len(completeness)} station-years meet >={MIN_COVERAGE:.0%} completeness")


(73040, 3) station-days with a valid daily mean: 29058
78 / 103 station-years meet >=70% completeness


In [4]:
daily_station["year"] = daily_station["date"].dt.year
keep_keys = set(zip(good_station_years["station"], good_station_years["year"]))
daily_station["keep"] = daily_station.apply(lambda r: (r["station"], r["year"]) in keep_keys, axis=1)
qualified = daily_station[daily_station["keep"]]
print("qualifying station-days:", qualified.shape)

network = qualified.groupby("date")["co"].mean().reset_index()
network = network.sort_values("date").reset_index(drop=True)
print(network.shape, network["date"].min(), network["date"].max())
print("non-null network-day count:", network["co"].notna().sum())
network.head()


qualifying station-days: (28481, 5)
(3652, 2) 2013-01-01 00:00:00 2022-12-31 00:00:00
non-null network-day count: 3650


,date,co
0,2013-01-01,0.479638
1,2013-01-02,0.695595
2,2013-01-03,0.821377
3,2013-01-04,0.845833
4,2013-01-05,0.761429


## Save

In [5]:
out_path = OUT_DIR / "bogota_co_daily.parquet"
network.to_parquet(out_path, index=False)
print("Saved:", out_path, network.shape)


Saved: /path/to/project/Data/processed/colombia/bogota/bogota_co_daily.parquet (3652, 2)
